In [ ]:
!mkdir /kaggle/temp

In [ ]:
!wget -O /kaggle/temp/Miniforge3.sh "https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-$(uname)-$(uname -m).sh"

In [ ]:
!rm -rf /kaggle/temp/conda
!bash /kaggle/temp/Miniforge3.sh -b -p /kaggle/temp/conda

In [ ]:
import subprocess
subprocess.run(['/kaggle/temp/conda/bin/mamba', 'install', '-y', 'python=3.13', 'cuda-version=13.0', 'pytorch=2.11',
                'pillow', 'matplotlib', 'scipy', 'nvimgcodec', 'nccl', 'cuda-cudart-dev'], check=True)
subprocess.run(['/kaggle/temp/conda/bin/python', '-c',
                'import sys, torch, triton; print(sys.version); print(torch.__version__, torch.version.cuda, torch.cuda.nccl.version(), triton.__version__)'],
               check=True)

In [ ]:
!rm -rf /kaggle/temp/music_deep
!git clone -b exp-034-dfine-multilayer-refinement https://github.com/kwon-young/music_deep.git /kaggle/temp/music_deep
!git -C /kaggle/temp/music_deep log --oneline -1

In [ ]:
import os
# The trompa-coco-cache pickle was built by the reverted Exp 032 code (SizeHistogram,
# 032 class weights), so the COCO JSON is parsed fresh (~1 min) with the current code.
os.makedirs('/kaggle/temp/cache', exist_ok=True)

# Locate the Exp 029 checkpoint in the attached model, wherever Kaggle mounts it.
print(sorted(os.listdir('/kaggle/input')))
wanted = os.path.join('029_size_difficulty_balanced_loss', 'train_detection', 'checkpoints')
found = []
for root, dirs, files in os.walk('/kaggle/input'):
    dirs[:] = [d for d in dirs if d not in ('datasets', 'ssl-imslp', 'trompa-coco', 'trompa-coco-cache')]
    if root.endswith(wanted) and 'latest_model.pt' in files:
        found.append(os.path.join(root, 'latest_model.pt'))
print('Exp 029 checkpoint:', found)
assert len(found) == 1, found
os.symlink(found[0], '/kaggle/temp/init_029.pt')

In [ ]:
!PYTHONPATH=/kaggle/temp/music_deep /kaggle/temp/conda/bin/mamba run timeout 11h torchrun --nproc_per_node=2 /kaggle/temp/music_deep/src/train_detection.py \
    --exp_dir experiments/034_dfine_multilayer_refinement \
    --patch_size 64 \
    --epochs 100 \
    --anno_path ../input/datasets/kwonyoungchoi/trompa-coco/annotations/instances_trainval2017.json \
    --img_dir ../input/datasets/kwonyoungchoi/trompa-coco/trainval2017 \
    --headless \
    --cache_dir /kaggle/temp/cache/ \
    --use_sdpa \
    --log_epoch_interval 5 \
    --warmup_epochs 1 \
    --lr 1e-4 \
    --compile \
    --extract_patch_min 54 \
    --extract_patch_max 74 \
    --class_weight_variance_lambda 0.5 \
    --class_weight_size_difficulty_max 8.0 \
    --refine_layers 2 \
    --dfine_outer_bins \
    --init_from_single_checkpoint /kaggle/temp/init_029.pt

In [ ]:
# Second session only, if `timeout 11h` stopped training before epoch 100: continue the same 100-epoch schedule.
# Replace --init_from_single_checkpoint with:
#   --resume --detector_checkpoint <previous output>/experiments/034_dfine_multilayer_refinement/train_detection/checkpoints/latest_model.pt

In [ ]:
!PYTHONPATH=/kaggle/temp/music_deep/src /kaggle/temp/conda/bin/mamba run python /kaggle/temp/music_deep/src/inference_coco.py \
    --checkpoint experiments/034_dfine_multilayer_refinement/train_detection/checkpoints/latest_model.pt \
    --output_dir experiments/034_dfine_multilayer_refinement/inference \
    --anno_path ../input/datasets/kwonyoungchoi/trompa-coco/annotations/instances_trainval2017.json \
    --img_dir ../input/datasets/kwonyoungchoi/trompa-coco/trainval2017 \
    --cache_dir /kaggle/temp/cache/ \
    --patch_size 64 \
    --use_sdpa \
    --eval_layer -1

In [ ]:
# Optional (time permitting): per-layer evaluation, tap 9 (layer 0) and tap 10 (layer 1)
!PYTHONPATH=/kaggle/temp/music_deep/src /kaggle/temp/conda/bin/mamba run python /kaggle/temp/music_deep/src/inference_coco.py \
    --checkpoint experiments/034_dfine_multilayer_refinement/train_detection/checkpoints/latest_model.pt \
    --output_dir experiments/034_dfine_multilayer_refinement/inference_layer0 \
    --anno_path ../input/datasets/kwonyoungchoi/trompa-coco/annotations/instances_trainval2017.json \
    --img_dir ../input/datasets/kwonyoungchoi/trompa-coco/trainval2017 \
    --cache_dir /kaggle/temp/cache/ \
    --patch_size 64 \
    --use_sdpa \
    --eval_layer 0

In [ ]:
!PYTHONPATH=/kaggle/temp/music_deep/src /kaggle/temp/conda/bin/mamba run python /kaggle/temp/music_deep/src/inference_coco.py \
    --checkpoint experiments/034_dfine_multilayer_refinement/train_detection/checkpoints/latest_model.pt \
    --output_dir experiments/034_dfine_multilayer_refinement/inference_layer1 \
    --anno_path ../input/datasets/kwonyoungchoi/trompa-coco/annotations/instances_trainval2017.json \
    --img_dir ../input/datasets/kwonyoungchoi/trompa-coco/trainval2017 \
    --cache_dir /kaggle/temp/cache/ \
    --patch_size 64 \
    --use_sdpa \
    --eval_layer 1